Sequential函数：实现网络层的快速串联

初始化模型参数：
1. 用正态分布初始化权重（`tensor.normal_(mean, std)`）
   - 权重不能全 0，否则每层输出都一样，模型无法学习
   - 用极小正态随机数初始化，梯度稳定，训练更容易收敛
2. 偏置全部初始化为 0 （`tensor.fill_(value)`）

SGD：PyTorch 内置的优化器，需要传入的参数：
1. 网络 net 里所有可训练参数
2. 学习率

In [13]:
# 实现线性回归
import numpy as np
import torch
from torch.utils import data
from d2l import torch as d2l
from torch import nn

true_w = torch.tensor([2,-3.4])
true_b = 4.2
features,labels = d2l.synthetic_data(true_w,true_b,1000) # 生成 1000 条带噪声的模拟数据集

def load_array(data_arrays, batch_size, is_train=True):
    """构造一个PyTorch数据迭代器：切分批次，自动配对特征 + 标签，自动打乱数据"""
    dataset = data.TensorDataset(*data_arrays)
    return data.DataLoader(dataset, batch_size, shuffle=is_train)

batch_size = 10
data_iter = load_array((features, labels), batch_size)

# define a model
net = nn.Sequential(nn.Linear(2,1)) # 第一个参数为输入的维度，第二个参数为输出的维度
net[0].weight.data.normal_(0,0.01)
net[0].bias.data.fill_(0)

# define loss function
loss = nn.MSELoss() # loss(预测值, 真实标签)：用均方误差 MSE 计算当前批次的损失值

# define optimization algorithm
trainer = torch.optim.SGD(net.parameters(), lr=0.03)

# train the model
num_epochs = 3 # 训练轮次
for epoch in range(num_epochs):
    for X, y in data_iter:
        l = loss(net(X), y)
        trainer.zero_grad() # 每一批计算梯度前，先清空上一轮残留梯度
        l.backward() # 算出损失l对网络所有权重w、偏置b的梯度
        trainer.step() # 梯度下降更新参数

print(net[0].weight.data)
print(net[0].bias.data)

tensor([[ 2.0003, -3.3996]])
tensor([4.1997])
